# 03 · Calibration

Choose the hedge ratio, signal and risk parameters for the chosen pair, using **TRAIN data only**,
and record them in `config.yaml`.

Sections: (a) hedge ratio, (b) entry/exit and z-score lookback, (c) risk rules, (d) costs.
Each section starts from the values currently in `config.yaml`; after every decision, save it and rerun the
notebook from the top so the following sections use it.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Make the repo root importable, whether Jupyter starts in notebooks/ or the repo root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import data, strategy, backtest, plots

plots.apply_style()
pd.set_option("display.float_format", "{:.4f}".format)
cfg = data.load_config()

In [ ]:
prices = data.load_prices(cfg, verbose=False)
funding = data.load_funding(cfg, coins=list(prices.columns))
pair = tuple(cfg["parameters"]["pair"])
params = data.get_params(cfg)

print(f"Pair: {pair[0]}/{pair[1]}  (long the spread = long {pair[0]}, short {pair[1]})")
print({k: v for k, v in params.items() if k != "train_end"})

## (a) Hedge ratio: static vs rolling

- **Static**: one beta from an OLS regression on the whole train period. Simple, but if the relationship drifts,
  the spread drifts too and the z-score keeps signalling in one direction.
- **Rolling**: beta re-estimated every hour on the last `hedge_window` hours (and the matching intercept, so the
  spread stays centred). Adapts to drift, but a short window also absorbs part of the mispricing we want to trade,
  and every beta change is a small rebalancing trade that costs fees.

In [ ]:
hedge_windows = {"static": None, "rolling 240h (10d)": 240, "rolling 720h (30d)": 720, "rolling 2160h (90d)": 2160}
hedge_results = {label: backtest.run_strategy(prices, funding, pair, data.get_params(cfg, hedge_window=window))
                 for label, window in hedge_windows.items()}
backtest.compare_metrics(hedge_results).loc[["sharpe", "total_return", "annual_return", "max_drawdown", "n_trades", "trades_per_month", "win_rate", "avg_holding_hours", "time_in_market", "total_fees", "total_slippage", "total_funding"]]

In [ ]:
display(plots.plot_series({label: res.beta for label, res in hedge_results.items()},
                          title=f"{pair[0]}/{pair[1]}: hedge ratio beta (train)", ylabel="beta"))
plots.plot_equity_curve(hedge_results, title=f"{pair[0]}/{pair[1]}: equity by hedge-ratio method (train)")

> ## 🟡 TEAM DECISION: (a) hedge ratio
>
> **Look at:** Sharpe and drawdown in the table, how much beta moves in the plot, and whether the equity curves
> differ consistently or only because of a few trades.
>
> **How to choose:** prefer the simplest method that is not clearly worse. If static is close to the best rolling
> window, static is easier to defend. If you choose rolling, prefer a window where neighbouring windows give similar
> results (robust), and remember that shorter windows mean more rebalancing costs. `null` = static.

In [ ]:
# Uncomment and edit after the team has decided, then rerun this notebook from the top:
# data.update_config({"hedge_window": 720})

## (b) Signal: entry threshold, z-score lookback and exit threshold

Sweep `entry_z` ∈ {1.5, 2.0, 2.5, 3.0} × `zscore_lookback` ∈ {24, 72, 168, 336} hours × `exit_z` ∈ {0, 0.5}
(32 backtests). The black frame marks the values currently in `config.yaml`.
Left: Sharpe (red = negative, blue = positive). Right: number of trades.

In [ ]:
signal_grid = {"entry_z": [1.5, 2.0, 2.5, 3.0], "zscore_lookback": [24, 72, 168, 336], "exit_z": [0.0, 0.5]}
signal_sweep = backtest.sweep(prices, funding, pair, params, signal_grid)
current = {"entry_z": params["entry_z"], "zscore_lookback": params["zscore_lookback"]}

for exit_z in signal_grid["exit_z"]:
    subset = signal_sweep[signal_sweep["exit_z"] == exit_z]
    chosen = current if exit_z == params["exit_z"] else None
    display(plots.plot_heatmap(subset, x="zscore_lookback", y="entry_z", values=["sharpe", "n_trades"],
                               chosen=chosen, title=f"{pair[0]}/{pair[1]} signal sweep, exit_z = {exit_z} (train)"))

In [ ]:
# Full sweep table, best Sharpe first
signal_sweep.sort_values("sharpe", ascending=False).head(10)

### Choose a plateau, not a peak

With 32 combinations, the single best cell is partly luck: it is the maximum of 32 noisy estimates and will
almost certainly look worse out of sample (**overfitting** / selection bias). A good choice sits in a **plateau**:
a region where the cell and its neighbours all perform similarly. Small changes of the parameters then do not
change the result much, which is what we expect from a real effect.

Also check the number of trades: fewer than ~30 trades on three years of data is too little to trust a Sharpe ratio,
and very many trades mean costs dominate (each round trip costs 2 × 7 bps of the traded notional).

> ## 🟡 TEAM DECISION: (b) entry_z, zscore_lookback, exit_z
>
> **Look at:** both heatmaps for both exit_z values. Find a region of similar, positive Sharpe with a reasonable
> number of trades.
>
> **How to choose:** pick a cell in the middle of that plateau, not the single best cell. If no region is
> positive, say so: that is a valid finding, and it tells us the pair or signal does not work, not that we should
> keep searching until something does.

In [ ]:
# Uncomment and edit after the team has decided, then rerun this notebook from the top:
# data.update_config({"entry_z": 2.0, "zscore_lookback": 168, "exit_z": 0.0})

## (c) Risk management

Compare the strategy with each risk rule switched on and off (all other parameters from `config.yaml`):

- **z-stop**: close when |z| > stop_z (the spread keeps diverging, maybe the relationship broke), then wait until |z| < entry_z.
- **time stop**: close after time_stop_halflives × half-life hours (the trade takes far longer than a normal reversion).
- **cointegration filter**: no new entries while the rolling cointegration p-value (90 days, updated daily) is above `coint_max_pvalue`.

If a rule never triggers, its variant is identical to the one without it and the two equity curves lie exactly on
top of each other (only the later one is visible). Check the exit-reason table to see which rules actually fire.

In [ ]:
stop_z = params["stop_z"] or 3.5                       # use the config value, or 3.5 if currently switched off
time_stop = params["time_stop_halflives"] or 2
risk_variants = {
    "no stops": dict(stop_z=None, time_stop_halflives=None, use_coint_filter=False),
    "z-stop only": dict(stop_z=stop_z, time_stop_halflives=None, use_coint_filter=False),
    "time stop only": dict(stop_z=None, time_stop_halflives=time_stop, use_coint_filter=False),
    "both stops": dict(stop_z=stop_z, time_stop_halflives=time_stop, use_coint_filter=False),
    "both + coint filter": dict(stop_z=stop_z, time_stop_halflives=time_stop, use_coint_filter=True),
}
risk_results = {label: backtest.run_strategy(prices, funding, pair, data.get_params(cfg, **overrides))
                for label, overrides in risk_variants.items()}
print(f"Half-life of the traded spread: {risk_results['no stops'].params['half_life']:.0f} h "
      f"-> time stop after {time_stop * risk_results['no stops'].params['half_life']:.0f} h")
backtest.compare_metrics(risk_results).loc[["sharpe", "total_return", "annual_return", "max_drawdown", "n_trades", "trades_per_month", "win_rate", "avg_holding_hours", "time_in_market", "total_fees", "total_slippage", "total_funding"]]

In [ ]:
# How each variant's trades ended
pd.DataFrame({label: res.trades["exit_reason"].value_counts() for label, res in risk_results.items()}).fillna(0).astype(int)

In [ ]:
display(plots.plot_equity_curve(risk_results, title=f"{pair[0]}/{pair[1]}: equity with and without risk rules (train)"))
plots.plot_drawdown(risk_results, title="Drawdown with and without risk rules (train)")

In [ ]:
# z-stop level: sweep stop_z (None = no z-stop), other parameters from config
stop_sweep = backtest.sweep(prices, funding, pair, params, {"stop_z": [3.0, 3.5, 4.0, None]})
stop_sweep

> ## 🟡 TEAM DECISION: (c) risk rules
>
> **Look at:** Sharpe **and** max drawdown in the table, the drawdown plot, and how many trades each rule closes.
>
> **How to choose:** a risk rule is not meant to raise the average return; it is meant to cut the worst losses.
> Keep a rule if it clearly reduces drawdowns or tail losses without destroying the Sharpe. Drop a rule that never
> triggers or only adds costs. Same plateau logic as in (b) for stop_z. `null` switches a rule off.

In [ ]:
# Uncomment and edit after the team has decided, then rerun this notebook from the top:
# data.update_config({"stop_z": 3.5, "time_stop_halflives": 2, "use_coint_filter": False})

## (d) Costs: where does the PnL come from?

Gross price PnL vs fees (0.05% per leg), slippage (0.02% per leg) and funding, for the parameters currently in
`config.yaml`. If the gross PnL is positive but smaller than the costs, the signal works but trades too often
or captures too little per trade.

In [ ]:
final = backtest.run_strategy(prices, funding, pair, params)
display(plots.plot_pnl_breakdown(final, title=f"{pair[0]}/{pair[1]}: PnL breakdown with config parameters (train)"))
display(plots.plot_trade_distribution(final))
pd.Series(final.metrics).to_frame("train")

**Rule of thumb:** one round trip costs about 2 × (0.05% + 0.02%) = **0.14% of the traded notional**
(entry and exit, both legs). The average trade PnL must clearly exceed that, otherwise the edge is eaten by costs.